# 🎨 Agentic Design Patterns with Azure AI Foundry Models (C#)

## 📋 Learning Objectives

This notebook demonstrates essential design patterns for building intelligent agents using the Microsoft Agent Framework with Azure AI Foundry (Azure OpenAI) model integration. You'll learn proven patterns and architectural approaches that make agents more robust, maintainable, and effective.

**Core Design Patterns Covered:**
- 🏗️ **Agent Factory Pattern**: Standardized agent creation and configuration
- 🔧 **Tool Registry Pattern**: Organized approach to managing agent capabilities
- 🧵 **Conversation Management**: Effective patterns for multi-turn interactions
- 🔄 **Response Processing**: Best practices for handling agent outputs

## 🎯 Key Architectural Concepts

### Design Principles
- **Separation of Concerns**: Clear boundaries between agent logic, tools, and configuration
- **Composability**: Building complex agents from reusable components
- **Extensibility**: Patterns that allow easy addition of new capabilities
- **Testability**: Design for easy unit testing and validation

### Azure AI Foundry Integration
- **Model Deployments**: Working with model deployment names (e.g., `gpt-4o`, `gpt-4o-mini`, `gpt-4.1-mini`) rather than raw model IDs
- **Endpoint Usage**: Calling Azure AI Foundry (Azure OpenAI) endpoints using Azure identity or API keys
- **Model Selection**: Choosing appropriate deployments for different use cases
- **Rate Limiting & Quotas**: Handling service constraints gracefully
- **Error Recovery**: Robust error handling and retry patterns

## 🔧 Technical Architecture

### Core Components
- **Microsoft Agent Framework**: C# implementation with Azure AI Foundry model support
- **Azure AI Foundry / Azure OpenAI API**: Access to state-of-the-art language models via secure endpoints
- **OpenAI-Compatible Client Pattern**: Standardized API interaction patterns (adapter approach)
- **Environment Configuration**: Secure and flexible configuration management

### Design Pattern Benefits
- **Maintainability**: Clear code organization and structure
- **Scalability**: Patterns that grow with your application needs
- **Reliability**: Proven approaches that handle edge cases
- **Performance**: Efficient resource utilization and API usage

## ⚙️ Prerequisites & Setup

**Required NuGet Packages:**
```bash
dotnet add package Microsoft.AgentFramework.Core
dotnet add package Azure.Identity
```

**Environment Configuration (.env file or User Secrets):**
```env
AZURE_AI_FOUNDRY_API_KEY=your_azure_openai_key
AZURE_AI_FOUNDRY_ENDPOINT=https://your-resource-name.openai.azure.com
AZURE_AI_FOUNDRY_MODEL=gpt-4o-mini
AZURE_OPENAI_API_VERSION=2024-08-01-preview
```
(Or configure Azure CLI / Managed Identity and use credential-based access.)

**Azure AI Foundry Access:**
- Azure subscription with Azure OpenAI access approved (if required)
- Deployed model (deployment name referenced by `AZURE_AI_FOUNDRY_MODEL`)
- Properly scoped API key OR Azure Active Directory auth
- Awareness of quota and rate limits for your pricing tier

## 📚 Design Pattern Categories

### 1. **Creational Patterns**
- Agent factory and builder patterns
- Configuration management patterns
- Dependency injection for agent services

### 2. **Behavioral Patterns**
- Tool execution and orchestration
- Conversation flow management  
- Response processing and formatting

### 3. **Integration Patterns**
- Azure AI Foundry endpoint integration
- Error handling and retry logic
- Resource management and cleanup

## 🚀 Best Practices Demonstrated

- **Clean Architecture**: Layered design with clear responsibilities
- **Error Handling**: Comprehensive exception management
- **Configuration**: Environment-based setup for different environments
- **Testing**: Patterns that enable effective unit and integration testing
- **Documentation**: Self-documenting code with clear intent

Ready to explore professional agent design patterns? Let's build something robust! 🌟

In [ ]:
// 📦 Install Required NuGet Packages
#r "nuget: Microsoft.AgentFramework.Core"
#r "nuget: Azure.Identity"

In [ ]:
// 📦 Import Core Libraries for Agent Design Patterns
using System;                      // Core system functionality
using System.Threading.Tasks;      // Asynchronous programming support
using System.Collections.Generic;  // Collection types

// Azure authentication and identity management
using Azure.Identity;

In [ ]:
// 🤖 Import Microsoft Agent Framework Components  
// ChatAgent: Core agent orchestration class following factory pattern
// AzureAIAgentClient / AzureOpenAIChatClient: Azure AI Foundry (Azure OpenAI) integration adapters
using Microsoft.AgentFramework;
using Microsoft.AgentFramework.Azure;

In [ ]:
// 🔧 Configuration Loading Pattern
// Loads environment variables for Azure AI Foundry model deployments and other config.
// Follows external configuration principle for cloud-native applications.

var modelDeploymentName = Environment.GetEnvironmentVariable("AZURE_AI_FOUNDRY_MODEL") ?? "gpt-4o";
var projectEndpoint = Environment.GetEnvironmentVariable("AZURE_AI_FOUNDRY_ENDPOINT") ?? "https://ibeljan-foundry.services.ai.azure.com/api/projects/firstProject";

Console.WriteLine($"Model Deployment: {modelDeploymentName}");
Console.WriteLine($"Project Endpoint: {projectEndpoint}");

In [ ]:
// 🛠️ Tool Function Design Pattern
// Implements the Strategy Pattern for pluggable agent capabilities.
// Demonstrates separation of business logic from agent orchestration.

string GetRandomDestination()
{
    /*
    Get a random vacation destination.
    
    Patterns illustrated:
    - Strategy Pattern: Interchangeable selection algorithm
    - Repository Pattern: Encapsulated data source
    - Factory Method: Creates destination objects on demand
    
    Returns:
        A randomly selected destination.
    */
    var destinations = new List<string>
    {
        "Barcelona, Spain",
        "Paris, France",
        "Berlin, Germany",
        "Tokyo, Japan",
        "Sydney, Australia",
        "New York, USA",
        "Cairo, Egypt",
        "Cape Town, South Africa",
        "Rio de Janeiro, Brazil",
        "Bali, Indonesia"
    };
    
    var random = new Random();
    return destinations[random.Next(0, destinations.Count)];
}

// Test the tool function
Console.WriteLine($"Random destination: {GetRandomDestination()}");

In [ ]:
// Initialize Azure AI Foundry chat client using a CLI credential (developer convenience).
// model_deployment_name should match a deployment you've created in Azure AI Foundry.

var credential = new AzureCliCredential();

var agentChatClient = new AzureAIAgentClient(
    credential: credential,
    modelDeploymentName: modelDeploymentName,
    projectEndpoint: new Uri(projectEndpoint)
);

Console.WriteLine("✅ Azure AI Agent Client created successfully");

In [ ]:
// 🤖 Agent Configuration Pattern
// Demonstrates externalized configuration and clear agent identity

const string AGENT_NAME = "TravelAgent";

const string AGENT_INSTRUCTIONS = @"You are a helpful AI Agent that can help plan vacations for customers.

Important: When users specify a destination, always plan for that location. Only suggest random destinations when the user hasn't specified a preference.

When the conversation begins, introduce yourself with this message:
""Hello! I'm your TravelAgent assistant. I can help plan vacations and suggest interesting destinations for you. Here are some things you can ask me:
1. Plan a day trip to a specific location
2. Suggest a random vacation destination
3. Find destinations with specific features (beaches, mountains, historical sites, etc.)
4. Plan an alternative trip if you don't like my first suggestion

What kind of trip would you like me to help you plan today?""

Always prioritize user preferences. If they mention a specific destination like ""Bali"" or ""Paris,"" focus your planning on that location rather than suggesting alternatives.
";

Console.WriteLine($"Agent Name: {AGENT_NAME}");
Console.WriteLine($"Instructions Length: {AGENT_INSTRUCTIONS.Length} characters");

In [ ]:
// Create the agent with Azure AI Foundry-backed chat client and a tool.
// Demonstrates the Agent Factory Pattern with dependency injection

var agent = new ChatAgent(
    name: AGENT_NAME,
    chatClient: agentChatClient,
    instructions: AGENT_INSTRUCTIONS,
    tools: new[] { GetRandomDestination }
);

Console.WriteLine($"✅ Agent '{AGENT_NAME}' created successfully with tools");

In [ ]:
// 🧵 Conversation Thread Pattern
// Creates a new conversation thread for maintaining context across multiple turns
// Demonstrates proper state management in conversational AI

var thread = agent.GetNewThread();

Console.WriteLine($"✅ New conversation thread created: {thread.Id}");

In [ ]:
// 🚀 First Agent Interaction
// Demonstrates async/await pattern for agent communication

var response1 = await agent.RunAsync("Plan me a day trip", thread: thread);

Console.WriteLine("✅ First response received");

In [ ]:
// 📖 Response Processing Pattern
// Demonstrates safe extraction and display of agent responses

var lastMessage = response1.Messages[response1.Messages.Count - 1];
var textContent = lastMessage.Contents[0].Text;

Console.WriteLine("Travel plan:");
Console.WriteLine(textContent);

In [ ]:
// 🔄 Multi-Turn Conversation Pattern
// Demonstrates context preservation across multiple agent interactions
// The agent remembers the previous destination and suggests a different one

var response2 = await agent.RunAsync("I don't like that destination. Plan me another vacation.", thread: thread);

Console.WriteLine("✅ Follow-up response received");

In [ ]:
// 📖 Display Alternative Travel Plan
// Shows how the agent handles context-aware follow-up requests

var lastMessage2 = response2.Messages[response2.Messages.Count - 1];
var textContent2 = lastMessage2.Contents[0].Text;

Console.WriteLine("Change plan:");
Console.WriteLine(textContent2);